In [9]:
from pymongo import MongoClient
from tqdm import tqdm
import pandas as pd

client = MongoClient("127.0.0.1", 27017)
db = client["bridge"]

In [10]:
def sample_records(df):
    res = []
    existing_old_old_fqns = set()
    for row in df.sample(frac=1, random_state=42).itertuples(index=False):
        fqns = set()
        for old_callee in row.old_callees:
            fqns.add(old_callee["full_name"])
        if fqns.issubset(existing_old_old_fqns):
            continue
        existing_old_old_fqns |= fqns
        res.append([row[0], row.library, row.old_version, row.new_version])
        if len(existing_old_old_fqns) >= 10:
            break
    return res


def sample_libraries(lang: str):
    data = []
    col = db[f"{lang}_api_call_changes"]
    for doc in tqdm(col.find({}), total=col.estimated_document_count()):
        version_before = doc["version_before"]
        version_after = doc["version_after"]
        parts_before = [int(_) for _ in version_before.split(".")]
        parts_after = [int(_) for _ in version_after.split(".")]
        if parts_before > parts_after:
            data.append(
                [
                    str(doc["_id"]),
                    doc["commit"],
                    doc["library"],
                    version_after,
                    version_before,
                    doc["new_callees"],
                    doc["old_callees"],
                ]
            )
        else:
            data.append(
                [
                    str(doc["_id"]),
                    doc["commit"],
                    doc["library"],
                    version_before,
                    version_after,
                    doc["old_callees"],
                    doc["new_callees"],
                ]
            )

    df = pd.DataFrame(
        data,
        columns=[
            "_id",
            "commit",
            "library",
            "old_version",
            "new_version",
            "old_callees",
            "new_callees",
        ],
    )
    df = df[df["old_callees"].str.len() * df["new_callees"].str.len() <= 25]
    count = df.groupby("library")["commit"].nunique()
    sampled_libraries = count.sample(n=100, weights=count.values, random_state=42).index
    record_samples = []
    for lib in sampled_libraries:
        record_samples.extend(sample_records(df[df["library"] == lib]))
    print(f"{lang}: {len(record_samples)} sampled records")
    pd.DataFrame(
        record_samples, columns=["_id", "library", "old_version", "new_version"]
    ).to_csv(f"../benchmark/{lang}_record_samples.csv", index=False)

In [3]:
sample_libraries("java")

100%|██████████| 342018/342018 [00:21<00:00, 16055.76it/s]


java: 638 sampled records


In [11]:
sample_libraries("py")

100%|██████████| 172540/172540 [00:13<00:00, 12571.98it/s]


py: 609 sampled records
